# 09. 프로덕션 배포 -- 테스트, 관측성, 배포

에이전트를 프로덕션에 배포하기 위한 전체 파이프라인을 다룹니다. 단위 테스트와 LangSmith 평가로 품질을 보장하고, 트레이싱으로 관측성을 확보한 뒤, LangGraph Platform으로 배포합니다.

```
개발 -> 테스트 (단위 + 평가) -> 관측성 (트레이싱) -> 배포 (LangGraph Platform)
```

### 왜 에이전트에 특화된 프로덕션 파이프라인이 필요한가?

에이전트는 전통적인 소프트웨어와 다른 특성을 갖습니다:

- **비결정적 실행**: 동일한 입력에도 다른 도구 호출 순서, 다른 응답을 생성할 수 있음
- **상태 유지**: 대화 기록과 체크포인트를 관리하는 장기 실행 프로세스
- **다중 컴포넌트**: LLM, 도구, 메모리, 체크포인터 등 여러 시스템이 연동

따라서 단순한 단위 테스트를 넘어 **궤적(trajectory) 평가**, **LLM-as-Judge**, **트레이스 기반 모니터링** 등 에이전트 전용 품질 보장 기법이 필요합니다.

## 학습 목표

이 노트북을 완료하면 다음을 수행할 수 있습니다:

1. **단위 테스트** -- `GenericFakeChatModel`로 LLM 응답을 모킹하여 결정적 테스트를 작성할 수 있다
2. **LangSmith 평가** -- 데이터셋 생성, 평가자 정의, 자동화된 에이전트 평가를 수행할 수 있다
3. **트레이스 분석** -- LangSmith 트레이싱으로 레이턴시, 토큰 사용량, 에러를 추적할 수 있다
4. **LangGraph Studio** -- 시각적 디버깅 도구로 에이전트 실행 흐름을 분석할 수 있다
5. **배포 옵션** -- LangGraph Platform, 셀프호스트, Cloud 배포 간 차이를 이해할 수 있다
6. **langgraph.json** -- 배포 설정 파일을 구성하고 배포 명령어를 실행할 수 있다

## 9.1 환경 설정

테스트, 관측성, 배포에 필요한 패키지를 설치합니다.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

import os

# Only enable LangSmith tracing if API key is available
if os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = "true"
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LANGSMITH_API_KEY 미설정. LangSmith 트레이싱 비활성화.")

In [ ]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault(
        "LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default")
    )
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler

    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")
# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


## 9.2 프로덕션 파이프라인 개요

에이전트의 비결정적 특성 때문에 전통적인 소프트웨어 테스트만으로는 품질을 보장할 수 없습니다.

| 단계 | 목적 | 도구 |
|------|------|------|
| **개발** | 에이전트 구현 및 로컬 테스트 | LangGraph, LangGraph Studio |
| **테스트** | 단위 테스트 + LLM 기반 평가 | pytest, agentevals, LangSmith |
| **관측성** | 트레이싱, 메트릭, 에러 추적 | LangSmith Tracing |
| **배포** | 프로덕션 환경 배포 | LangGraph Platform |

### 테스트 전략

에이전트 테스트는 두 가지 접근을 병행해야 합니다:

| 유형 | 특징 | 적합 대상 |
|------|------|----------|
| **단위 테스트** | `GenericFakeChatModel`로 LLM 응답을 모킹하여 격리된 결정적 테스트 수행. API 호출 없이 빠르게 실행 | 개별 도구, 파서, 프롬프트 포맷팅, 상태 변환 로직 |
| **통합 테스트** | 실제 네트워크 호출로 컴포넌트 간 협업 검증. `agentevals`의 궤적 평가로 에이전트 행동 패턴 분석 | 전체 에이전트 흐름, 도구 호출 시퀀스, 최종 응답 품질 |

에이전트 시스템은 다중 컴포넌트가 체이닝되어 동작하므로, 일반적으로 통합 테스트의 비중이 더 높습니다. 단위 테스트로 개별 컴포넌트의 정확성을 확인하고, 통합 테스트로 전체 흐름의 품질을 평가합니다.

## 9.3 에이전트 테스트 -- LangSmith 평가

`agentevals` 패키지는 에이전트 궤적(trajectory) 전용 평가자를 제공합니다. 궤적이란 에이전트가 최종 응답에 도달하기까지의 모든 단계(도구 호출, 중간 추론, 의사결정)를 의미합니다.

| 전략 | 설명 | 장점 | 단점 |
|------|------|------|------|
| **Trajectory Match** | 기대 시퀀스와 단계별 비교. 미리 정의한 참조 궤적과 에이전트의 실제 궤적을 매칭 | 정확한 검증, 재현 가능 | 구체적 기대값 필요, 유연성 낮음 |
| **LLM-as-Judge** | LLM이 궤적을 루브릭(평가 기준) 기반으로 정성 평가. 도구 사용 적절성, 응답 정확성 등을 자동 판단 | 유연한 평가, 복잡한 시나리오 대응 | 추가 LLM 비용, 평가 자체의 비결정성 |

### Trajectory Match 모드

에이전트의 도구 호출 순서에 대한 기대 수준을 4단계로 조절할 수 있습니다:

| 모드 | 설명 | 사용 시점 |
|------|------|----------|
| `strict` | 메시지와 도구 호출 순서가 참조와 완전히 동일해야 통과 | 순서가 중요한 워크플로 (예: 인증 -> 조회 -> 업데이트) |
| `unordered` | 동일한 도구들을 호출했으면 순서 무관하게 통과 | 독립적인 도구 호출이 여러 개인 경우 |
| `subset` | 에이전트가 참조 도구만 호출 (추가 도구 호출 없음) | 불필요한 도구 호출을 방지하고 싶을 때 |
| `superset` | 에이전트가 참조 도구를 최소한 포함 (추가 호출 허용) | 핵심 도구 호출만 보장하고 싶을 때 |

In [ ]:
try:
    from langsmith import Client

    ls_client = Client()

    dataset = ls_client.create_dataset("agent-eval-v1")
    ls_client.create_examples(
        inputs=[{"query": "LangGraph란 무엇인가요?"}],
        outputs=[{"expected": "에이전트를 위한 프레임워크"}],
        dataset_id=dataset.id,
    )
    print("데이터셋 생성됨:", dataset.name)
except Exception as e:
    print(f"LangSmith 미설정 (건너뜀): {e}")
    ls_client = None
    dataset = None

In [ ]:
try:
    from agentevals.trajectory import create_trajectory_llm_as_judge

    evaluator = create_trajectory_llm_as_judge(
        rubric=("에이전트가 적절한 도구를 사용했습니까? 최종 답변이 정확했습니까?"),
    )
    print("평가자 생성됨:", type(evaluator).__name__)
except ImportError:
    print("agentevals 미설치. 설치: pip install agentevals")
    evaluator = None
except Exception as e:
    print(f"평가자 생성 건너뜀 (LLM API 키 필요): {e}")
    evaluator = None

## 9.4 단위 테스트 패턴

`GenericFakeChatModel`을 사용하면 API 호출 없이 LLM 응답을 모킹하여 결정적 테스트를 작성할 수 있습니다. 응답 이터레이터를 받아 호출마다 하나씩 반환합니다.

### 왜 모킹이 필요한가?

에이전트 테스트에서 실제 LLM API를 호출하면 다음 문제가 발생합니다:
- **비결정적 결과**: 동일 입력에도 매번 다른 응답이 올 수 있어 테스트 재현이 어려움
- **비용**: 테스트를 실행할 때마다 API 비용 발생
- **속도**: 네트워크 지연으로 테스트 속도 저하
- **가용성**: API 장애 시 테스트 실패

`GenericFakeChatModel`은 미리 정의한 응답을 순서대로 반환하므로, **결정적이고 빠르며 무료인** 테스트를 작성할 수 있습니다. 스트리밍 패턴도 지원하여 `astream()` 기반 코드도 테스트 가능합니다.

### 상태 지속성 테스트

`InMemorySaver` 체크포인터를 사용하면 여러 대화 턴에 걸친 상태 의존적 행동을 테스트할 수 있습니다. `thread_id`를 지정하여 같은 대화 컨텍스트를 유지하면서 여러 호출의 누적 상태를 검증합니다.

In [ ]:
from langchain_core.language_models import GenericFakeChatModel
from langchain_core.messages import AIMessage

fake_responses = [
    AIMessage(content="LangGraph는 프레임워크입니다."),
    AIMessage(content="상태 기반 에이전트를 지원합니다."),
]
fake_model = GenericFakeChatModel(messages=iter(fake_responses))
print(fake_model.invoke("LangGraph란 무엇인가요?", config=lf_config).content)

In [ ]:
def search_tool(query: str) -> str:
    """웹에서 정보를 검색합니다."""
    return f"검색 결과: {query}"


def test_search_tool():
    """검색 도구가 예상 형식을 반환하는지 테스트합니다."""
    result = search_tool("test query")
    assert isinstance(result, str) and len(result) > 0
    print("통과: test_search_tool")


test_search_tool()

### HTTP 요청 녹화/재생

CI/CD에서 API 비용을 줄이기 위해 `vcrpy`와 `pytest-recording`으로 HTTP 요청을 녹화하고 재생할 수 있습니다. 첫 실행 시 실제 API 호출을 녹화(cassette 파일로 저장)하고, 이후 실행에서는 녹화된 응답을 재생하여 네트워크 호출 없이 테스트합니다.

이 방식의 장점:
- **첫 실행**: 실제 API와의 통합을 검증 (통합 테스트 역할)
- **이후 실행**: 녹화된 응답으로 빠르고 결정적인 테스트 (단위 테스트 역할)
- **CI/CD**: API 키 없이도 테스트 실행 가능

```python
import pytest

@pytest.fixture(scope="module")
def vcr_config():
    return {"record_mode": "once"}

@pytest.mark.vcr()
def test_agent_with_recorded_responses():
    result = agent.invoke("What is LangGraph?")
    assert "framework" in result.lower()
```

## 9.5 관측성 -- LangSmith 트레이싱

LangSmith 트레이싱은 에이전트 실행의 **모든 단계**를 기록합니다. 트레이스(trace)는 초기 사용자 입력부터 최종 응답까지, 모든 모델 호출, 도구 사용, 의사결정 포인트를 포함하는 에이전트 실행의 완전한 기록입니다.

### 트레이스에 기록되는 정보

| 항목 | 설명 |
|------|------|
| **입력/출력** | 각 단계의 입력 데이터와 출력 결과 |
| **모델 호출** | 프롬프트, 응답, 모델 파라미터 |
| **도구 호출** | 호출된 도구, 인자, 반환값 |
| **레이턴시** | 각 단계별 소요 시간 |
| **토큰 사용량** | 입력/출력 토큰 수 |
| **에러** | 실패한 단계와 에러 메시지 |

### 활성화 방법

환경 변수 2개만 설정하면 **추가 코드 없이** 자동 트레이싱됩니다:

```bash
export LANGSMITH_TRACING=true
export LANGSMITH_API_KEY=<your-api-key>
```

`create_agent`로 생성한 에이전트는 환경 변수 설정 시 자동으로 실행 데이터를 LangSmith에 전송합니다. LangChain의 모든 컴포넌트(LLM, 체인, 도구 등)가 내장 계측(instrumentation)을 포함하고 있어 별도의 코드 수정이 필요 없습니다.

### 선택적 트레이싱

`tracing_context`를 사용하면 특정 코드 블록만 선택적으로 트레이싱할 수 있습니다. 이를 통해 디버깅이 필요한 부분만 집중적으로 모니터링하거나, 프로젝트별로 트레이스를 분리할 수 있습니다.

In [ ]:
print("LANGSMITH_TRACING:", os.environ.get("LANGSMITH_TRACING"))
print("LANGSMITH_PROJECT:", os.environ.get("LANGSMITH_PROJECT"))

try:
    from langsmith import tracing_context

    with tracing_context(project_name="debug-session"):
        print("이 블록에 대해 트레이싱 활성화됨")
except Exception as e:
    print(f"LangSmith 트레이싱 사용 불가: {e}")

## 9.6 트레이스 분석

LangSmith 대시보드에서 트레이스를 분석하여 프로덕션 에이전트의 품질을 모니터링합니다.

| 메트릭 | 설명 | 확인 포인트 |
|--------|------|-------------|
| **Latency** | 각 단계별 소요 시간 | 병목 구간 식별 (어느 노드가 가장 느린지) |
| **Token Usage** | 입력/출력 토큰 수 | 비용 최적화 (프롬프트 길이 조절, 불필요한 컨텍스트 제거) |
| **Error Rate** | 실패한 실행 비율 | 안정성 모니터링 (특정 도구의 실패율, LLM 타임아웃 등) |
| **Tool Call Frequency** | 도구별 호출 빈도 | 에이전트 행동 패턴 분석 (과도한 도구 호출, 미사용 도구 식별) |

### 태그와 메타데이터 활용

`config` 파라미터나 `tracing_context`로 커스텀 태그와 메타데이터를 추가하여 트레이스를 분류하고 필터링할 수 있습니다. 프로덕션 환경에서 유용한 태그/메타데이터 예시:

| 태그/메타데이터 | 용도 |
|----------------|------|
| **버전 태그** (`v2.1`) | A/B 테스트, 버전별 성능 비교 |
| **실험 태그** (`experiment-A`) | 프롬프트 변경 등 실험 추적 |
| **사용자 티어** (`premium`) | 사용자 그룹별 품질 모니터링 |
| **리전** (`kr`) | 지역별 레이턴시 분석 |

### 프로젝트 관리

프로젝트는 두 가지 방식으로 설정할 수 있습니다:
- **정적**: `LANGSMITH_PROJECT` 환경 변수로 기본 프로젝트 지정
- **동적**: `tracing_context(project_name=...)`으로 코드 블록별 프로젝트 분리

In [ ]:
try:
    from langsmith import tracing_context

    with tracing_context(
        project_name="production-agent",
        tags=["v2.1", "experiment-A"],
        metadata={"user_tier": "premium", "region": "kr"},
    ):
        print("태그된 트레이싱 활성화됨")
except Exception as e:
    print(f"LangSmith 트레이싱 사용 불가: {e}")

## 9.7 LangGraph Studio -- 시각적 디버깅

LangGraph Studio는 에이전트의 실행 흐름을 **시각적으로** 디버깅할 수 있는 무료 도구입니다. 로컬 머신에서 에이전트를 개발하고 테스트하는 데 최적화되어 있습니다.

| 기능 | 설명 |
|------|------|
| **그래프 시각화** | 에이전트의 노드/엣지 구조를 실시간 확인. 현재 실행 중인 노드를 하이라이트 |
| **단계별 실행** | 각 노드의 입출력 데이터를 검사하며 디버깅. 프롬프트, 도구 호출, 결과를 단계별로 확인 |
| **상태 검사** | 에이전트의 전체 상태를 시각적으로 탐색. 메시지 히스토리, 체크포인트 데이터 포함 |
| **실시간 스트리밍** | 에이전트 실행 과정을 실시간으로 관찰. 토큰/레이턴시 메트릭 제공 |

### 로컬 개발 서버 설정

Studio를 사용하려면 LangGraph CLI로 로컬 개발 서버를 시작합니다:

```bash
# LangGraph CLI 설치 (Python 3.11+ 필요)
pip install --upgrade "langgraph-cli[inmem]"

# 개발 서버 시작
langgraph dev
```

서버가 시작되면 `https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024` 에서 Studio UI에 접근할 수 있습니다.

### Studio에서 확인 가능한 정보

- 에이전트에게 전송되는 프롬프트
- 각 도구 호출과 그 결과
- 최종 출력
- 중간 상태 (검사 및 수정 가능)
- 토큰 사용량과 레이턴시 메트릭

## 9.8 배포 옵션

에이전트는 **상태를 유지하는 장기 실행 프로세스**이므로 일반 웹앱 호스팅과 다른 접근이 필요합니다. 전통적인 stateless 웹 애플리케이션 호스팅(예: Vercel, Heroku)은 에이전트의 지속적 상태 관리, 백그라운드 실행, 체크포인팅에 적합하지 않습니다.

| 옵션 | 설명 | 적합 대상 |
|------|------|----------|
| **LangGraph Cloud** | LangSmith 관리형 호스팅. GitHub 연결만으로 자동 빌드/배포 | 빠른 프로토타이핑, 소규모 팀 |
| **Self-Hosted** | 자체 인프라에서 Docker 컨테이너로 실행 | 데이터 주권, 엔터프라이즈, 규제 환경 |
| **Hybrid** | 클라우드 관리 + 자체 런타임 | 관리 편의성 + 데이터 제어 양립 |

### 배포 요구사항

| 항목 | 설명 |
|------|------|
| **GitHub 저장소** | 코드 호스팅 (공개/비공개 모두 지원) |
| **LangSmith 계정** | 무료 가입 가능 (smith.langchain.com) |
| **langgraph.json** | 의존성, 그래프, 환경 변수를 정의하는 배포 설정 파일 |

### LangGraph Cloud 배포 프로세스

1. LangSmith에 로그인 후 Deployments 페이지로 이동
2. "+ New Deployment" 클릭
3. GitHub 계정 연결 (비공개 저장소의 경우)
4. 저장소 선택 후 제출 (약 15분 소요)
5. 배포 완료 후 API URL 복사하여 클라이언트에서 사용

## 9.9 langgraph.json 설정

`langgraph.json`은 배포의 핵심 설정 파일입니다. 의존성, 그래프 엔트리포인트, 환경 변수를 정의합니다. 이 파일이 프로젝트 루트에 있어야 LangGraph CLI와 Cloud 배포가 정상 동작합니다.

In [ ]:
import json

langgraph_config = {
    "dependencies": ["."],
    "graphs": {"agent": "./src/agent.py:graph"},
    "env": ".env",
}
print(json.dumps(langgraph_config, indent=2))

### 설정 항목 상세

| 항목 | 타입 | 설명 |
|------|------|------|
| `dependencies` | `list[str]` | Python 패키지 의존성. `.`은 현재 디렉터리의 `pyproject.toml`을 참조 |
| `graphs` | `dict` | 그래프 이름과 모듈 경로 매핑. `"모듈경로:변수명"` 형식 |
| `env` | `str` | 환경 변수 파일 경로 (`.env` 형식). API 키 등 민감 정보 포함 |

### 그래프 엔트리포인트

`graphs` 필드의 값은 `"./경로/파일.py:변수명"` 형식입니다. 변수는 `CompiledGraph` 인스턴스여야 합니다. LangGraph의 Graph API(`StateGraph`)와 Functional API(`@entrypoint`) 모두 사용 가능합니다.

#### Graph API 예시

```python
# src/agent.py
from langgraph.prebuilt import create_react_agent

graph = create_react_agent(
    model="claude-sonnet-4-6",
    tools=[search_tool],
    checkpointer=True,
)
```

#### Functional API 예시

LangGraph의 Functional API를 사용하면 기존 Python 코드에 최소한의 변경으로 영속성, 메모리, 스트리밍을 통합할 수 있습니다. `@entrypoint` 데코레이터가 워크플로의 시작점을 정의하고, `@task` 데코레이터가 개별 작업 단위를 나타냅니다.

```python
# src/agent.py
from langgraph.func import entrypoint, task

@task
def process_query(query: str) -> str:
    return f"처리 완료: {query}"

@entrypoint(checkpointer=checkpointer)
def graph(inputs: dict) -> str:
    result = process_query(inputs["query"])
    return result.result()
```

Functional API의 핵심 특징:
- **표준 Python 제어 흐름** 사용 (if/for 등) -- 명시적 그래프 구조 불필요
- **함수 스코프 상태 관리** -- 별도의 State 선언이나 reducer 설정 불필요
- **태스크 결과 체크포인팅** -- 재실행 시 이전에 완료된 태스크 결과를 자동 재사용
- **입출력 JSON 직렬화 필수** -- 체크포인터 사용 시 모든 데이터가 직렬화 가능해야 함

## 9.10 배포 명령어

LangGraph CLI를 사용한 빌드, 로컬 서버, 클라우드 배포 명령어입니다.

### 1. Docker 이미지 빌드

`langgraph.json` 설정을 기반으로 Docker 이미지를 생성합니다:

```bash
langgraph build -t my-agent:latest
```

### 2. 로컬 서버 실행

로컬에서 에이전트를 실행하여 테스트합니다. Studio UI와 연동하여 시각적 디버깅이 가능합니다:

```bash
# 프로덕션 모드 (Docker 기반)
langgraph up --config langgraph.json

# 개발 모드 (인메모리, 빠른 시작)
langgraph dev
```

### 3. 클라우드 배포

LangSmith 대시보드에서:
1. Deployments -> "+ New Deployment"
2. GitHub 저장소 연결
3. 리포지토리 선택 후 제출 (약 15분 소요)
4. 배포 완료 후 API URL 복사

### Python SDK로 배포된 에이전트 접근

`langgraph-sdk`를 사용하면 배포된 에이전트와 프로그래밍 방식으로 통신할 수 있습니다. 스트리밍, 스레드 관리, 상태 조회 등 모든 기능을 SDK로 제어합니다.

In [ ]:
try:
    from langgraph_sdk import get_sync_client

    api_key = os.environ.get("LANGSMITH_API_KEY", "")
    if not api_key:
        print("LANGSMITH_API_KEY 미설정. 클라이언트 연결 건너뜀.")
    else:
        client = get_sync_client(
            url="https://your-deployment.langsmith.com",
            api_key=api_key,
        )
        print("클라이언트 연결됨:", type(client).__name__)
except Exception as e:
    print(f"LangGraph SDK 클라이언트 사용 불가: {e}")

### REST API로 접근

배포된 에이전트는 REST API로도 접근할 수 있습니다. 이를 통해 어떤 프로그래밍 언어/프레임워크에서도 에이전트와 통신 가능합니다:

```bash
curl --request POST \
  --url <DEPLOYMENT_URL>/runs/stream \
  --header 'Content-Type: application/json' \
  --header 'X-Api-Key: <LANGSMITH_API_KEY>' \
  --data '{
    "assistant_id": "agent",
    "input": {
      "messages": [
        {"role": "user", "content": "안녕하세요!"}
      ]
    },
    "stream_mode": "updates"
  }'
```

주요 엔드포인트:

| 엔드포인트 | 메서드 | 설명 |
|-----------|--------|------|
| `/runs/stream` | POST | 스트리밍 실행 |
| `/runs` | POST | 동기 실행 |
| `/threads` | POST | 새 스레드 생성 |
| `/threads/{id}/state` | GET | 스레드 상태 조회 |

## 요약

| 주제 | 핵심 내용 |
|------|----------|
| **테스트 전략** | 단위 테스트(`GenericFakeChatModel`) + 통합 테스트(`agentevals`) 병행 |
| **LangSmith 평가** | Trajectory Match (strict/unordered/subset/superset), LLM-as-Judge |
| **관측성** | `LANGSMITH_TRACING=true` + `LANGSMITH_API_KEY`로 자동 트레이싱 |
| **트레이스 분석** | Latency, Token Usage, Error Rate, Tool Call Frequency |
| **LangGraph Studio** | 시각적 그래프 디버깅, 단계별 상태 검사 |
| **배포 옵션** | Cloud (관리형), Self-Hosted (Docker), Hybrid |
| **langgraph.json** | `dependencies`, `graphs`, `env` 3가지 핵심 설정 |
| **배포 명령어** | `langgraph build` -> `langgraph up` -> LangSmith Deploy |
